In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from pff import reports_dir
from pff.models.utils.callbacks_refactor import (
    _predictive_images_per_batch,
    _vpc_images_per_batch,
    sample_predictive_bundle,
    sample_vpc_bundle,
)
from pff.training.basic_experiment import BasicLightningExperiment


In [4]:
experiment = BasicLightningExperiment.from_experiment_dir("/home/cesarali/Pharma/pff/results/faithful_valance_6141_")
#experiment = BasicLightningExperiment.from_experiment_comet(
#    "2a98913dcef6444caf29fc0ee4ce3843"
#)

# Option 1: load from a local experiment directory
# experiment = BasicLightningExperiment.from_experiment_dir(
#     "/home/cesarali/Pharma/pff/results/comet/functional-flow-pk/e19fb57257cd4501bcdb3a2294dc559e"
# )

# Option 2: load from Comet experiment id


#
model = experiment.model
dm = experiment.datamodule
dm.prepare_data()
dm.setup()
model_device = getattr(model, "device", None)
if not isinstance(model_device, torch.device):
    model_device = next(model.parameters()).device

/home/cesarali/Pharma/pff/pff/models/utils/loss_utils.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.weights = torch.tensor(weights if weights else torch.ones(number_of_losses))
CometLogger will be initialized in online mode


# Plot VPC


In [15]:
selected_drug = "midazolam"
empirical_batches = dm.get_empirical_test_batches(no_heldout=True, device=model_device)

available_studies, available_drugs = dm.describe_empirical_test_batches(
    empirical_batches=empirical_batches,
    no_heldout=True,
    print_available=True,
)


Available empirical datasets (no_heldout=True): ['cesarali/lenuzza-2016', 'cesarali/Indometacin', 'cesarali/Theophylline']
Dataset 'cesarali/lenuzza-2016' contains 1 empirical batch(es).
  Batch 0 studies: ['Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016']
  Batch 0 drugs: ['memantine', 'omeprazole', '5-hydroxyomeprazole', 'omeprazole sulfone', 'repaglinide', 'hydroxy repaglinide', 'rosuvastatin', 'tolbutamide', '4-hydroxytolbutamide', 'dextromethorphan', 'digoxin', 'paracetamol', '1-hydroxymidazolam', 'paracetamol glucuronide', 'dextrorphan', 'caffeine (137X)', 'midazolam', 'paraxanthine (17X)']
Dataset 'cesarali/Indometacin' contains 1 empirical batch(es).
  Batch 0 studies: ['Indometacin']
  Batch 0 drugs: ['Indometacin']
Dataset 'cesarali/Theophylline' contains 1 em

In [16]:
single_drug_batch, selected_study_name, selected_drug_name = dm.select_empirical_drug_batch(
    empirical_batches=empirical_batches,
    selected_drug=selected_drug,
    print_selection=True,
)


Selected empirical dataset key: cesarali/lenuzza-2016
Selected empirical batch index: 0
Selected study: Lenuzza2016
Selected drug: midazolam


In [5]:
#single_drug_batch

In [17]:
sample_size = 500
vpc_bundle = sample_vpc_bundle(model, single_drug_batch, sample_size=sample_size)

Sampling trajectories (new individual):   0%|            | 0/50 [00:00<?, ?it/s]

Sampling trajectories (new individual): 100%|███| 50/50 [00:01<00:00, 36.17it/s]


In [19]:
output_root = reports_dir / "notebooks_vpc"
image_paths = _vpc_images_per_batch(
    bundle=vpc_bundle,
    batch=single_drug_batch,
    label="Empirical",
    epoch=0,
    output_root=output_root,
    model_label="faithful_valance",
    n_bins=10,
    binning="equal_count",
    log_y=False,
)

if not image_paths:
    raise RuntimeError("No VPC image was produced for the selected drug.")

single_vpc_path = Path(image_paths[0])
print("Saved VPC image:", single_vpc_path)


Saved VPC image: /home/cesarali/Pharma/pff/reports/notebooks_vpc/faithful_valance/vpc/empirical/epoch_000_midazolam_000.png


/home/cesarali/Pharma/pff/pff/metrics/sampling_quality.py:273: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  observed_values.groupby("Time")["Value"]
/home/cesarali/Pharma/pff/pff/metrics/sampling_quality.py:281: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  predicted_values.groupby(["Time", "Replicate"])["Value"]
/home/cesarali/Pharma/pff/pff/metrics/sampling_quality.py:286: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
 

# Plot Prediction


In [5]:
import importlib
import pff.utils.plots.databatch_plot as databatch_plot
import pff.models.utils.callbacks_refactor as callbacks_refactor

importlib.reload(databatch_plot)
importlib.reload(callbacks_refactor)
sample_predictive_bundle = callbacks_refactor.sample_predictive_bundle
_predictive_images_per_batch = callbacks_refactor._predictive_images_per_batch


In [6]:
heldout_batches = dm.get_empirical_test_batches(no_heldout=False, device=model_device)

heldout_available_studies, heldout_available_drugs = dm.describe_empirical_test_batches(
    empirical_batches=heldout_batches,
    no_heldout=False,
    print_available=True,
)


Available empirical datasets (heldout): ['cesarali/lenuzza-2016', 'cesarali/Indometacin', 'cesarali/Theophylline']
Dataset 'cesarali/lenuzza-2016' contains 10 empirical batch(es).
  Batch 0 studies: ['Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016', 'Lenuzza2016']
  Batch 0 drugs: ['memantine', 'omeprazole', '5-hydroxyomeprazole', 'omeprazole sulfone', 'repaglinide', 'hydroxy repaglinide', 'rosuvastatin', 'tolbutamide', '4-hydroxytolbutamide', 'dextromethorphan', 'digoxin', 'paracetamol', '1-hydroxymidazolam', 'paracetamol glucuronide', 'dextrorphan', 'caffeine (137X)', 'midazolam', 'paraxanthine (17X)']
Dataset 'cesarali/Indometacin' contains 6 empirical batch(es).
  Batch 0 studies: ['Indometacin']
  Batch 0 drugs: ['Indometacin']
Dataset 'cesarali/Theophylline' contains 12 empirica

In [15]:
len(heldout_batches['cesarali/lenuzza-2016'])

10

# Do One

In [7]:
permutation_indexes = [0,1,2,3,4,8]
drugs_list = ['memantine', 'omeprazole', '5-hydroxyomeprazole', 'omeprazole sulfone', 'repaglinide', 'hydroxy repaglinide', 'rosuvastatin', 'tolbutamide', '4-hydroxytolbutamide', 'dextromethorphan', 'digoxin', 'paracetamol', '1-hydroxymidazolam', 'paracetamol glucuronide', 'dextrorphan', 'caffeine (137X)', 'midazolam', 'paraxanthine (17X)', 'Indometacin', 'Theophylline']
selected_drug = 'midazolam'
heldout_single_drug_batch_list, heldout_selected_study, heldout_selected_drug = dm.select_empirical_drug_batch(
    empirical_batches=heldout_batches,
    selected_drug=selected_drug,
    print_selection=True,
    permutation_indexes=permutation_indexes
)

heldout_single_drug_batch = heldout_single_drug_batch_list[0]
predictive_bundle = sample_predictive_bundle(model, heldout_single_drug_batch)


Selected empirical dataset key: cesarali/lenuzza-2016
Selected empirical batch indexes: [0, 1, 2, 3, 4, 8]
Selected study: Lenuzza2016
Selected drug: midazolam


Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 36.59
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 48.67
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 48.02
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 45.83
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 48.25
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 44.89
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 45.91
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 45.34
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 44.73
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 42.14


In [ ]:


output_root = reports_dir / "notebooks_vpc"

predictive_plot_kwargs = {
    "figure_size": (8, 6),
    "title": selected_drug.capitalize(),
    "title_font_size": 19,
    "show_legend": True,
    "legend_font_size": 12,
    "axis_label_font_size": 18,
    "tick_label_font_size": 14,
    "point_size": 32,
    "point_marker": "o",
    "prediction_marker": "o",
    "prediction_marker_size": 9,
    "context_obs_color": "forestgreen",
    "context_rem_color": "mediumseagreen",
}

predictive_image_paths = _predictive_images_per_batch(
    bundle=predictive_bundle,
    batch=heldout_single_drug_batch,
    model=model,
    label="Empirical",
    epoch=0,
    perm_index=0,
    output_root=output_root,
    model_label="loaded_model",
    plot_kwargs=predictive_plot_kwargs,
)


single_predictive_path = Path(predictive_image_paths[0])
print("Saved predictive image:", single_predictive_path)


Saved predictive image: /home/cesarali/Pharma/pff/reports/notebooks_vpc/loaded_model/predictions/empirical/epoch_000_perm_000_midazolam_permutation_0.png


# Do a Loop

In [9]:

import importlib
import pff.utils.plots.databatch_plot as databatch_plot
import pff.models.utils.callbacks_refactor as callbacks_refactor

importlib.reload(databatch_plot)
importlib.reload(callbacks_refactor)
sample_predictive_bundle = callbacks_refactor.sample_predictive_bundle
_predictive_images_per_batch = callbacks_refactor._predictive_images_per_batch


permutation_indexes = [2,3]
drugs_list = ['memantine', 'omeprazole', '5-hydroxyomeprazole', 'omeprazole sulfone', 'repaglinide', 'hydroxy repaglinide', 'rosuvastatin', 'tolbutamide', '4-hydroxytolbutamide', 'dextromethorphan', 'digoxin', 'paracetamol', '1-hydroxymidazolam', 'paracetamol glucuronide', 'dextrorphan', 'caffeine (137X)', 'midazolam', 'paraxanthine (17X)', 'Indometacin', 'Theophylline']
drugs_list = ['memantine','Indometacin', 'Theophylline', 'caffeine (137X)', 'midazolam', 'paraxanthine (17X)']
for selected_drug in drugs_list:
    heldout_single_drug_batch_list, heldout_selected_study, heldout_selected_drug = dm.select_empirical_drug_batch(
        empirical_batches=heldout_batches,
        selected_drug=selected_drug,
        print_selection=True,
        permutation_indexes=permutation_indexes
    )

    heldout_single_drug_batch = heldout_single_drug_batch_list[0]
    predictive_bundle = sample_predictive_bundle(model, heldout_single_drug_batch)

    for j in range(len(permutation_indexes)):
        heldout_single_drug_batch = heldout_single_drug_batch_list[j]
        predictive_bundle = sample_predictive_bundle(model, heldout_single_drug_batch)


        output_root = reports_dir / "notebooks_vpc"

        predictive_plot_kwargs = {
            "figure_size": (8, 6),
            "title": selected_drug.capitalize(),
            "title_font_size": 19,
            "show_legend": True,
            "legend_font_size": 12,
            "axis_label_font_size": 18,
            "tick_label_font_size": 14,
            "point_size": 32,
            "point_marker": "o",
            "prediction_marker": "o",
            "prediction_marker_size": 9,
            "context_obs_color": "forestgreen",
            "context_rem_color": "mediumseagreen",
        }

        predictive_image_paths = _predictive_images_per_batch(
            bundle=predictive_bundle,
            batch=heldout_single_drug_batch,
            model=model,
            label="Empirical",
            epoch=0,
            perm_index=j,
            output_root=output_root,
            model_label="loaded_model",
            plot_kwargs=predictive_plot_kwargs,
        )

        if not predictive_image_paths:
            raise RuntimeError("No predictive image was produced for the selected heldout drug.")

        single_predictive_path = Path(predictive_image_paths[0])
        print("Saved predictive image:", single_predictive_path)


Selected empirical dataset key: cesarali/lenuzza-2016
Selected empirical batch indexes: [2, 3]
Selected study: Lenuzza2016
Selected drug: memantine


Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 33.44
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 48.34
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 44.96
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 44.27
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 48.07
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 47.33
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 46.87
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 42.87
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 46.53
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 43.76
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 45.06
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 46.41
Sampling trajectories (indiv

Saved predictive image: /home/cesarali/Pharma/pff/reports/notebooks_vpc/loaded_model/predictions/empirical/epoch_000_perm_000_memantine_permutation_0.png


Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 44.96
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 44.61
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 44.04
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 44.03
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 44.66
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 47.02
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 48.45
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 46.64
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 46.53
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 47.34


Saved predictive image: /home/cesarali/Pharma/pff/reports/notebooks_vpc/loaded_model/predictions/empirical/epoch_000_perm_001_memantine_permutation_0.png
Selected empirical dataset key: cesarali/Indometacin
Selected empirical batch indexes: [2, 3]
Selected study: Indometacin
Selected drug: Indometacin


...

Saved predictive image: /home/cesarali/Pharma/pff/reports/notebooks_vpc/loaded_model/predictions/empirical/epoch_000_perm_001_midazolam_permutation_0.png
Selected empirical dataset key: cesarali/lenuzza-2016
Selected empirical batch indexes: [2, 3]
Selected study: Lenuzza2016
Selected drug: paraxanthine (17X)


Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 46.83
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:00<00:00, 50.33
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 48.91
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:00<00:00, 52.79
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 48.79
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:00<00:00, -2693
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:00<00:00, 50.89
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 43.50
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 49.66
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 49.85
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 45.12
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:00<00:00, 51.81
Sampling trajectories (indiv

Saved predictive image: /home/cesarali/Pharma/pff/reports/notebooks_vpc/loaded_model/predictions/empirical/epoch_000_perm_000_paraxanthine_17X_permutation_0.png


Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 47.30
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 49.02
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 48.77
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 34.46
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 35.89
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 44.26
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 42.49
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 31.63
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:01<00:00, 49.44
Sampling trajectories (individual prediction): 100%|█| 50/50 [00:00<00:00, 51.56


Saved predictive image: /home/cesarali/Pharma/pff/reports/notebooks_vpc/loaded_model/predictions/empirical/epoch_000_perm_001_paraxanthine_17X_permutation_0.png


In [10]:
predictive_bundle.samples_S.shape

torch.Size([500, 1, 1, 10, 1])